*Fake News Detection Model*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import joblib
from collections import Counter

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

stop_words = set(stopwords.words('english'))

In [ ]:
fake = pd.read_csv("datasets/Fake.csv")
true = pd.read_csv("datasets/True.csv")
print('Fake news shape:', fake.shape)
print('Real news shape:', true.shape)

In [ ]:
fake["label"] = 1
true["label"] = 0

In [ ]:
data = pd.concat([fake, true], ignore_index=True)
print('Combined dataset shape:', data.shape)

In [ ]:
sns.countplot(x='label', data=data)
plt.title('Fake vs Real News Distribution')
plt.xticks([0, 1], ['Real', 'Fake'])
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

In [ ]:
data['text_length'] = data['text'].apply(len)
sns.histplot(data=data, x='text_length', hue='label', bins=50)
plt.title('Article Length Distribution')
plt.xlabel('Article Length')
plt.ylabel('Count')
plt.show()

In [ ]:
fake_words = ' '.join(data[data['label'] == 1]['text']).split()
real_words = ' '.join(data[data['label'] == 0]['text']).split()

fake_freq = Counter(fake_words).most_common(20)
real_freq = Counter(real_words).most_common(20)

fake_df = pd.DataFrame(fake_freq, columns=['word', 'count'])
real_df = pd.DataFrame(real_freq, columns=['word', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x='count', y='word', data=fake_df, ax=axes[0])
axes[0].set_title('Top 20 Words in Fake News')
sns.barplot(x='count', y='word', data=real_df, ax=axes[1])
axes[1].set_title('Top 20 Words in Real News')
plt.show()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

In [ ]:
data['text'] = data['text'].apply(clean_text)
print('Text cleaning complete!')

In [ ]:
X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print('Training set size:', X_train_tfidf.shape)
print('Testing set size:', X_test_tfidf.shape)

In [ ]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)
print('Model training complete!')

In [ ]:
y_pred = model.predict(X_test_tfidf)
print('Accuracy:', accuracy_score(y_test, y_pred))

In [ ]:
cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=5)
print('CV Scores:', cv_scores)
print('Mean Accuracy:', cv_scores.mean())
print('Standard Deviation:', cv_scores.std())

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

In [ ]:
model = joblib.load('fake_news_model.pkl')
vectorizer = joblib.load('vectorizer.pkl')

sample = ["i am cool"]
sample_tfidf = vectorizer.transform(sample)
prediction = model.predict(sample_tfidf)
print('Prediction:', 'Fake' if prediction[0] == 1 else 'Real')

In [ ]:
joblib.dump(model, 'fake_news_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print('Model and vectorizer saved!')